In [1]:
import networkx as graphs
from itertools import combinations
from collections import Counter, defaultdict
import pandas as pd
import requests
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stopwords

In [2]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

DOC_TITLES = [
    "Pizza",
    "Pasta",
    "Burrata",
    "Danny Sanderson",
    "Gidi Gov",
    "Yehudit Ravitz",
    "Apple Inc.",
    "Intel",
    "IBM",
    "Flute",
    "Violin",
    "Double Bass",
    "Raspberry",
    "Strawberry",
    "Blueberry"
]


HEADERS = {
    "User-Agent": "Python/requests"
}

In [3]:
def wiki(title):
    params = {
        "action": "query",
        "prop": "extracts|info",
        "explaintext": False,
        "titles": title,
        "inprop" : "url",
        "format": "json",
        "redirects": 0,
        "formatversion": 2,
        "exintro" : 1
    }
    response = requests.get(WIKI_API, params=params, headers=HEADERS)
    pages = response.json().get("query", {}).get("pages", [])
    if not pages:
        return ""
    return pages[0].get("extract", "") ,pages[0].get("canonicalurl",""), pages[0].get("title","")

In [4]:
def buildgraph(text):
    tokens =[t for t in (re.split(r"\W+", text.lower())) if t.isalpha() and t not in stopwords and len(t)>1]
    graph = graphs.DiGraph() #  coocurences are considered as one directional (ltr), hence the use of a directed graph
    for i in range(len(tokens)-1):
        if tokens[i]!=tokens[i+1]:
            if graph.has_edge(tokens[i],tokens[i+1]):
                graph[tokens[i]][tokens[i+1]]["weight"]+=1
            else:
                graph.add_edge(tokens[i],tokens[i+1],weight=1)
    return graph

In [5]:
def scoregraph(graph):
    scores = graphs.pagerank(graph, weight = 'weight') #since TextRank is essentialy a special case of PageRank, we can apply networkx's built-in PageRank and consider the assigned weights
    rankings = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return rankings

<h4>all of the snippets above were fetched from previous assignments and slightly modified to comply with the requirements of this assignment</h4>

In [6]:
def tenkeywordsperdoc(titles):
    docs = {}
    keywordsperdoc = []
    for i, input_title in enumerate(titles, start=1):
        print(f"Fetching: {input_title}")
        text, url, title = wiki(input_title)
        docs[i] = {"title": title, "text": text}
        keywordsperdoc.append({(docs[i]["title"],url) : [x[0] for x in scoregraph(buildgraph(text))[:10]]})
    for item in keywordsperdoc:
        titleslst = list(item.keys())
        print(f"10 first keywords in page with title {titleslst[0][0]} are: {', '.join(kws for kws in item[titleslst[0]])}\n")
    return keywordsperdoc

<h4>This function builds a list of dictionaries with the 10 first keywords in each page</h4>

In [7]:
def build_inverted_index(docs):
    inverted = defaultdict(list)
    for url, text in docs.items():
        tokens =[t for t in (re.split(r"\W+", text.lower())) if t.isalpha() and t not in stopwords and len(t)>1]
        uniqueterms = set(tokens)
        for term in uniqueterms:
            posting = {"url": url}
            inverted[term].append(posting)
    return dict(inverted)

<h4>This function builds an inverted index based on the list obtained from the 'tenkeywordsperdoc' list</h4>

In [8]:
keywordsperdoc = tenkeywordsperdoc(DOC_TITLES)
keywordslst = []
for item in keywordsperdoc:
    keywordslst.extend(item[list(item.keys())[0]])

Fetching: Pizza
Fetching: Pasta
Fetching: Burrata
Fetching: Danny Sanderson
Fetching: Gidi Gov
Fetching: Yehudit Ravitz
Fetching: Apple Inc.
Fetching: Intel
Fetching: IBM
Fetching: Flute
Fetching: Violin
Fetching: Double Bass
Fetching: Raspberry
Fetching: Strawberry
Fetching: Blueberry
10 first keywords in page with title Pizza are: pizza, oven, world, eaten, typically, dish, neapolitan, italian, pizzerias, restaurants

10 first keywords in page with title Pasta are: pasta, shapes, dishes, fresh, italian, produced, dried, pastas, cooked, names

10 first keywords in page with title Burrata are: cream, cheese, region, italy, dish, born, milk, need, minimise, food

10 first keywords in page with title Danny Sanderson are: israeli, sanderson, guitarist, songwriter, singer, november, contribution, musician, music, born

10 first keywords in page with title Gidi Gov are: gov, anat, playwright, screenwriter, married, actor, presenter, television, singer, israeli

10 first keywords in page wit

In [9]:
docs = []
for intitle in DOC_TITLES:
    print(f"Fetching: {intitle}")
    text,url,title = wiki(intitle)
    docs.append({"url": url, "text": text})
doc_texts = {info["url"]: info["text"] for info in docs}
print("\nBuilding Inverted Index...\n")
index = build_inverted_index(doc_texts)
for term in keywordslst:
    postings = index.get(term.lower(), [])
    print(f"{len(postings)} postings found for {term}")
    for p in postings:
        print(p["url"])

Fetching: Pizza
Fetching: Pasta
Fetching: Burrata
Fetching: Danny Sanderson
Fetching: Gidi Gov
Fetching: Yehudit Ravitz
Fetching: Apple Inc.
Fetching: Intel
Fetching: IBM
Fetching: Flute
Fetching: Violin
Fetching: Double Bass
Fetching: Raspberry
Fetching: Strawberry
Fetching: Blueberry

Building Inverted Index...

1 postings found for pizza
https://en.wikipedia.org/wiki/Pizza
2 postings found for oven
https://en.wikipedia.org/wiki/Pizza
https://en.wikipedia.org/wiki/Pasta
7 postings found for world
https://en.wikipedia.org/wiki/Pizza
https://en.wikipedia.org/wiki/Apple_Inc.
https://en.wikipedia.org/wiki/Intel
https://en.wikipedia.org/wiki/IBM
https://en.wikipedia.org/wiki/Raspberry
https://en.wikipedia.org/wiki/Strawberry
https://en.wikipedia.org/wiki/Blueberry
2 postings found for eaten
https://en.wikipedia.org/wiki/Pizza
https://en.wikipedia.org/wiki/Strawberry
3 postings found for typically
https://en.wikipedia.org/wiki/Pizza
https://en.wikipedia.org/wiki/Pasta
https://en.wikipedia.

<h4>This function fetches the documents using Wikipedia's API and prints an inverted index for the first 10 keywords in each page, taking all the fetched documents as the corpora</h4>